# VideoShield records for three MOSE samples

VideoShield embeds its watermark in a diffusion generator's initial noise. The MOSE clips already exist, so an authentic VideoShield watermark cannot be added by post-processing them.

Each sample therefore contains an unwatermarked negative-control detection record. There is no VideoShield-watermarked video, removal output, predicted mask, or IoU for these clips. Those fields are not applicable because watermark embedding never occurred.


## Metric guide

| Term | Meaning | Interpretation |
| --- | --- | --- |
| Negative-control bit accuracy | Fraction of reference watermark bits matched in an unwatermarked input. | About 0.5 is chance for balanced binary bits. It is not an embedding or robustness score. |
| DDIM inversion steps | Number of reverse-diffusion steps used by the detector control. | A configuration value, not a quality metric. |
| Not applicable | The experiment cannot be performed faithfully for an existing MOSE clip. | It is different from failure and different from a score of zero. |
| Runtime | Wall-clock time recorded on the experiment hardware. | Useful for this run, not a hardware-independent benchmark. |
| SHA-256 | File fingerprint stored in the JSON record. | Confirms exact file identity. It is not a quality metric. |


## Notebook setup

After a fresh clone, run `python3 scripts/prepare_mose_notebooks.py` from the repository root. This downloads and verifies the 48 videos shared by the three method notebooks. Run the next cell once to enable the video, image, download, and metadata viewers. If Git LFS has left a pointer file in place of a video, the viewer reports the preparation command instead of displaying a broken player.


In [ ]:
import base64
from pathlib import Path
import html
import json
from IPython.display import HTML, Markdown, display

LFS_POINTER_PREFIX = b"version https://git-lfs.github.com/spec/v1"

def find_repo_root():
    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (start / "evidence/mose-task1/review50/manifest.json").is_file():
            return start
    raise FileNotFoundError("Run this notebook from inside the paper-reproductions checkout.")

REPO_ROOT = find_repo_root()
REVIEW_ROOT = REPO_ROOT / "evidence/mose-task1/review50"

def artifact_data_url(source, mime_type):
    if not source.is_file():
        raise FileNotFoundError(source)
    payload = source.read_bytes()
    if payload.startswith(LFS_POINTER_PREFIX):
        raise RuntimeError(
            f"{source} is a Git LFS pointer, not the real media file. "
            "From the repository root run: python3 scripts/prepare_mose_notebooks.py"
        )
    return f"data:{mime_type};base64,{base64.b64encode(payload).decode('ascii')}"

def show_videos(items, columns=3):
    cards = []
    for title, relative, explanation in items:
        if relative is None:
            cards.append(
                f"<section class='media-card'><h4>{html.escape(title)}</h4>"
                f"<p><strong>Not applicable.</strong></p><p>{html.escape(explanation)}</p></section>"
            )
            continue
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "video/mp4")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<video controls muted loop playsinline preload='metadata' src='{data_url}'></video>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this video</a></p>"
            "</section>"
        )
    display(HTML(
        f"<style>.media-grid{{display:grid;grid-template-columns:repeat({columns},minmax(220px,1fr));gap:16px}}"
        ".media-card{border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#fff;color:#24292f}"
        ".media-card video{width:100%;background:#000}.media-card h4{margin:0 0 8px}.media-card p{font-size:13px}</style>"
        f"<div class='media-grid'>{''.join(cards)}</div>"
    ))

def show_images(items, columns=2):
    cards = []
    for title, relative, explanation in items:
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "image/png")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<img style='width:100%' src='{data_url}' alt='{html.escape(title)}'>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this image</a></p>"
            "</section>"
        )
    display(HTML(
        f"<div class='media-grid' style='grid-template-columns:repeat({columns},minmax(260px,1fr))'>"
        f"{''.join(cards)}</div>"
    ))

def show_metadata(paths):
    for relative in paths:
        path = REVIEW_ROOT / relative
        payload = json.loads(path.read_text())
        display(Markdown(f"#### `{relative}`"))
        print(json.dumps(payload, indent=2))

print(f"Loaded evidence from {REVIEW_ROOT}")


## 1. Sample `02b72b8d`

The source has 67 frames at 960 x 480 and 25 FPS. The original and raw mask are shown as context. They are not VideoShield outputs.

The negative control sampled 16 frames, resized them to 256 x 256, and used 25 DDIM inversion steps. The reference watermark was not embedded into the clip.

| Metric or configuration | Value | Interpretation |
| --- | ---: | --- |
| Negative-control bit accuracy | **0.5000** (50.00%) | 0.00 percentage points from the 50% chance level. |
| Sampled frames | 16 | Evenly selected control frames from the source. |
| Detector resize | 256 x 256 | Resolution used by the detector control. |
| DDIM inversion steps | 25 | Detector configuration. |
| Control runtime | 3.817 s | Runtime for the negative control. |
| Watermarked video | N/A | No authentic VideoShield embedding exists for this clip. |
| ProPainter and DiffuEraser outputs | N/A | No watermarked input exists to attack. |
| Predicted tamper mask and IoU | N/A | No valid removal experiment exists for this clip. |

**Interpretation:** A value near 0.5 is expected for an unwatermarked input. This control does not measure watermark robustness or localization.


In [ ]:
show_videos([('Original MOSE video', 'media/02b72b8d/original.mp4', 'Unwatermarked source used for the negative control.'), ('Raw selected-object mask', 'media/02b72b8d/gt_mask.mp4', 'Removal region shown only as source context.'), ('VideoShield watermarked and removal outputs', None, 'Authentic VideoShield embedding requires diffusion generation, so these outputs do not exist for this clip.')])


### Availability and control record

The next cell prints the exact sampling configuration, control result, runtime, interpretation, and explicit not-applicable fields.


In [ ]:
show_metadata(['media/02b72b8d/methods/videoshield/availability.json'])


## 2. Sample `0ab75563`

The source has 104 frames at 960 x 448 and 25 FPS. The original and raw mask are shown as context. They are not VideoShield outputs.

The negative control sampled 16 frames, resized them to 256 x 256, and used 25 DDIM inversion steps. The reference watermark was not embedded into the clip.

| Metric or configuration | Value | Interpretation |
| --- | ---: | --- |
| Negative-control bit accuracy | **0.4805** (48.05%) | 1.95 percentage points from the 50% chance level. |
| Sampled frames | 16 | Evenly selected control frames from the source. |
| Detector resize | 256 x 256 | Resolution used by the detector control. |
| DDIM inversion steps | 25 | Detector configuration. |
| Control runtime | 3.815 s | Runtime for the negative control. |
| Watermarked video | N/A | No authentic VideoShield embedding exists for this clip. |
| ProPainter and DiffuEraser outputs | N/A | No watermarked input exists to attack. |
| Predicted tamper mask and IoU | N/A | No valid removal experiment exists for this clip. |

**Interpretation:** A value near 0.5 is expected for an unwatermarked input. This control does not measure watermark robustness or localization.


In [ ]:
show_videos([('Original MOSE video', 'media/0ab75563/original.mp4', 'Unwatermarked source used for the negative control.'), ('Raw selected-object mask', 'media/0ab75563/gt_mask.mp4', 'Removal region shown only as source context.'), ('VideoShield watermarked and removal outputs', None, 'Authentic VideoShield embedding requires diffusion generation, so these outputs do not exist for this clip.')])


### Availability and control record

The next cell prints the exact sampling configuration, control result, runtime, interpretation, and explicit not-applicable fields.


In [ ]:
show_metadata(['media/0ab75563/methods/videoshield/availability.json'])


## 3. Sample `0b02886a`

The source has 30 frames at 960 x 640 and 25 FPS. The original and raw mask are shown as context. They are not VideoShield outputs.

The negative control sampled 16 frames, resized them to 256 x 256, and used 25 DDIM inversion steps. The reference watermark was not embedded into the clip.

| Metric or configuration | Value | Interpretation |
| --- | ---: | --- |
| Negative-control bit accuracy | **0.4980** (49.80%) | 0.20 percentage points from the 50% chance level. |
| Sampled frames | 16 | Evenly selected control frames from the source. |
| Detector resize | 256 x 256 | Resolution used by the detector control. |
| DDIM inversion steps | 25 | Detector configuration. |
| Control runtime | 3.401 s | Runtime for the negative control. |
| Watermarked video | N/A | No authentic VideoShield embedding exists for this clip. |
| ProPainter and DiffuEraser outputs | N/A | No watermarked input exists to attack. |
| Predicted tamper mask and IoU | N/A | No valid removal experiment exists for this clip. |

**Interpretation:** A value near 0.5 is expected for an unwatermarked input. This control does not measure watermark robustness or localization.


In [ ]:
show_videos([('Original MOSE video', 'media/0b02886a/original.mp4', 'Unwatermarked source used for the negative control.'), ('Raw selected-object mask', 'media/0b02886a/gt_mask.mp4', 'Removal region shown only as source context.'), ('VideoShield watermarked and removal outputs', None, 'Authentic VideoShield embedding requires diffusion generation, so these outputs do not exist for this clip.')])


### Availability and control record

The next cell prints the exact sampling configuration, control result, runtime, interpretation, and explicit not-applicable fields.


In [ ]:
show_metadata(['media/0b02886a/methods/videoshield/availability.json'])


## Validation and provenance

- The committed validator decoded the videos, checked frame counts, dimensions, and FPS, and verified the required method fields.
- Aggregate and per-file validation: [validation.json](../../evidence/mose-task1/review50/validation.json)
- Aggregate results: [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json)
- Experimental contract: [PROTOCOL.md](../../evidence/mose-task1/PROTOCOL.md)
- Batch summary: [README.md](../../evidence/mose-task1/README.md)

The displayed values come from the committed VideoShield evidence records.
